# Model Design
----

## Review
- Datasets, Dataloaders and Transforms
- We mightt want to perform additional transformation for a more sensible features
  - feature transformation to get `home_size=total_rooms/households`, `home_density=total_bedrooms/total_rooms`,`household_size=population/households`

In [7]:
import torch
from torch import nn
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from pathlib import Path

### Feature Engineering
- Feature Transformation
- Normalization
- Feature Selection

In [19]:
from sklearn.preprocessing import StandardScaler
import os
import numpy as np

In [9]:
  def feature_transform(df):
    # this internal method will serve as transform method for both the features and atarges
    df['home_size'] = df['total_rooms']/df['households']
    df['home_density'] = df['total_bedrooms']/df['total_rooms']
    df['household_size'] = df['population']/df['households']
    return df[['longitude','latitude','housing_median_age','median_income','home_size','home_density','household_size','median_house_value']]

In [31]:
dataset_path = Path("/content/sample_data")

train_df = pd.read_csv(dataset_path/"california_housing_train.csv")
test_df = pd.read_csv(dataset_path/"california_housing_test.csv")

# Feature Transform
train_df = feature_transform(train_df)
test_df = feature_transform(test_df)

print(test_df.columns)


# Normalization (we accept outliers for now)
## Fit scaler to train only to avoid data leakage
scaler = StandardScaler()
scaler.fit(train_df)

train_np = scaler.transform(train_df)
test_np = scaler.transform(test_df)

# Save as preprocessed data
PREPROCESSED_DIR = Path("/content/preprocessed_data")
os.makedirs(name=PREPROCESSED_DIR, exist_ok=True)
np.savez_compressed(PREPROCESSED_DIR/"train.npz", data=train_np, )
np.savez_compressed(PREPROCESSED_DIR/"test.npz", data=test_np)


Index(['longitude', 'latitude', 'housing_median_age', 'median_income',
       'home_size', 'home_density', 'household_size', 'median_house_value'],
      dtype='object')


In [63]:
# test load
# with np.load(PREPROCESSED_DIR/"train.npz") as data:
#     X = data['data']
#     print(X[:, :8].shape)

In [60]:
## Dataset

def to_tensor(x):
  return torch.from_numpy(x)

class HousingDataset(Dataset):
  def __init__(self, root: Path, train: bool = True, transform=None, target_transform=None):
    with np.load(PREPROCESSED_DIR/f"{'train' if train else 'test' }.npz") as data:
        X,y = data['data'][:,:7], data['data'][:,7:8]

    if transform:
      self.X = transform(X)
    if target_transform:
      self.y = transform(y)


  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx,], self.y[idx,]


train_dataset = HousingDataset(dataset_path, train=True, transform=to_tensor, target_transform=to_tensor)
test_dataset = HousingDataset(dataset_path, train=False, transform=to_tensor, target_transform=to_tensor)

In [62]:
# for idx, (X, y) in enumerate(test_dataset):
#   print(idx)
#   print(X.shape, X)
#   print(y.shape, y)
#   break

In [64]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

In [65]:
for X,y in train_dataloader:
  print(X.shape, y.shape)
  break

torch.Size([32, 7]) torch.Size([32, 1])


## Building a Neural Network
Here we want to build a `Regression` model that identifies the `median house value` of a house based on final features.